# QaTa-COV19 All-in-One Kaggle Notebook

Notebook này đã chứa toàn bộ code cần thiết trong chính file `.ipynb`, không cần import các file Python khác trong repo.

Cách dùng:
1. Upload đúng 1 notebook này lên Kaggle.
2. Add dataset QaTa-COV19 đã đúng format LViT.
3. Bật GPU và Internet.
4. Nếu dùng đúng dataset của bạn thì có thể để nguyên config mặc định.
5. Chỉnh `LABEL_RATIO` ở cell config thành `1.0`, `0.5` hoặc `0.25` khi cần.
6. Chạy `Run All`.

In [ ]:
!pip uninstall -y -q torch torchvision torchaudio
!pip install -q torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q ml-collections openpyxl protobuf sentencepiece thop timm "transformers<5"

import torch
print('torch version:', torch.__version__)
print('torch cuda:', torch.version.cuda)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('supported arch:', torch.cuda.get_arch_list())
    x = torch.randn(1, device='cuda')
    print('cuda smoke tensor:', x)

In [ ]:
DATASET_ROOT = "/kaggle/input/datasets/tqc0103/qata-covid19/QaTa-Covid19"
TASK_NAME = "QaTa-COV19"
LABEL_RATIO = 1.0  # change to 0.5 or 0.25 on other Kaggle accounts
SAVE_DIR = f"/kaggle/working/qatacov19_{int(LABEL_RATIO * 100):03d}pct"

EPOCHS = 120
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
IMAGE_SIZE = 224
LR = 1e-4
MIN_LR = 1e-6
WEIGHT_DECAY = 1e-4
SCHEDULER = "cosine"
SEED = 666
NUM_WORKERS = 2
EMA_DECAY = 0.99
TEXT_MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
MAX_TEXT_UNITS = 10
LOCAL_FILES_ONLY = False
AMP = False
SAVE_EVERY = 0
SAVE_VAL_EXAMPLES = 4

print('Effective batch size =', BATCH_SIZE * GRAD_ACCUM_STEPS)
print('Outputs will be saved to', SAVE_DIR)

In [ ]:
import sys
import types


def register_module(module_name: str, source: str, package: str | None = None):
    module = types.ModuleType(module_name)
    module.__file__ = module_name.replace('.', '/') + '.py'
    module.__package__ = package if package is not None else module_name.rpartition('.')[0]
    sys.modules[module_name] = module
    exec(compile(source, module.__file__, "exec"), module.__dict__)
    return module


nets_pkg = types.ModuleType("nets")
nets_pkg.__path__ = []
sys.modules["nets"] = nets_pkg

UTILS_SOURCE = "import numpy as np\nimport pandas as pd\n\n\ndef dice_coef(y_true, y_pred):\n    smooth = 1e-5\n    y_true_f = y_true.flatten()\n    y_pred_f = y_pred.flatten()\n    intersection = np.sum(y_true_f * y_pred_f)\n    return (2.0 * intersection + smooth) / (np.sum(y_true_f) + np.sum(y_pred_f) + smooth)\n\n\ndef dice_on_batch(masks, pred):\n    dices = []\n    for i in range(pred.shape[0]):\n        pred_tmp = pred[i][0].cpu().detach().numpy()\n        mask_tmp = masks[i].cpu().detach().numpy()\n        pred_tmp[pred_tmp >= 0.5] = 1\n        pred_tmp[pred_tmp < 0.5] = 0\n        mask_tmp[mask_tmp > 0] = 1\n        mask_tmp[mask_tmp <= 0] = 0\n        dices.append(dice_coef(mask_tmp, pred_tmp))\n    return np.mean(dices)\n\n\ndef read_text(filename):\n    df = pd.read_excel(filename, engine='openpyxl')\n    text = {}\n    for i in df.index.values:\n        count = len(df.Description[i].split())\n        if count < 9:\n            df.loc[i, 'Description'] = df.Description[i] + ' EOF XXX' * (9 - count)\n        text[df.Image[i]] = df.Description[i]\n    return text\n"
TEXT_ENCODER_SOURCE = 'import re\nimport threading\nfrom pathlib import Path\nfrom typing import Any, Iterable, List, Sequence\n\nimport torch\n\n\nFEATURE_CACHE_FORMAT_VERSION = 3\nREPORT_UNIT_SPLIT_VERSION = 2\n\nATTRIBUTE_GROUPS = {\n    "laterality": ["unknown", "left", "right", "bilateral", "diffuse"],\n    "vertical": ["unknown", "upper", "middle", "lower", "basal"],\n    "count": ["unknown", "single", "multiple", "diffuse"],\n    "extent": ["unknown", "focal", "multifocal", "diffuse"],\n    "severity": ["unknown", "mild", "moderate", "severe"],\n}\n\n\ndef attribute_vector_size() -> int:\n    return sum(len(options) for options in ATTRIBUTE_GROUPS.values())\n\n\ndef build_cache_metadata(\n    model_name: str,\n    max_units: int,\n    parser_name: str = "structured-report-parser-v1",\n) -> dict[str, Any]:\n    return {\n        "format_version": FEATURE_CACHE_FORMAT_VERSION,\n        "model_name": model_name,\n        "max_units": int(max_units),\n        "parser_name": parser_name,\n        "report_unit_split_version": REPORT_UNIT_SPLIT_VERSION,\n    }\n\n\ndef split_report_into_units(report: str) -> List[str]:\n    text = (report or "").strip()\n    if not text:\n        return ["[NO_TEXT]"]\n\n    raw_units = re.split(r"[\\n.;]+|,\\s+(?=[A-Za-z])", text)\n    units = [unit.strip() for unit in raw_units if unit and unit.strip()]\n    if not units:\n        return ["[NO_TEXT]"]\n    return units\n\n\ndef feature_cache_path(cache_dir: str | Path, sample_id: str) -> Path:\n    stem = Path(sample_id).stem\n    return Path(cache_dir) / f"{stem}.pt"\n\n\ndef save_report_features(\n    cache_dir: str | Path,\n    sample_id: str,\n    text: torch.Tensor,\n    attributes: torch.Tensor,\n    metadata: dict[str, Any] | None = None,\n) -> Path:\n    cache_path = feature_cache_path(cache_dir, sample_id)\n    cache_path.parent.mkdir(parents=True, exist_ok=True)\n    payload = {\n        "text": text.cpu(),\n        "attributes": attributes.cpu(),\n        "metadata": metadata or {"format_version": FEATURE_CACHE_FORMAT_VERSION},\n    }\n    torch.save(payload, cache_path)\n    return cache_path\n\n\ndef _metadata_matches(\n    cached_metadata: dict[str, Any] | None,\n    expected_metadata: dict[str, Any] | None,\n) -> bool:\n    if expected_metadata is None:\n        return True\n    if cached_metadata is None:\n        return False\n    for key, expected_value in expected_metadata.items():\n        if cached_metadata.get(key) != expected_value:\n            return False\n    return True\n\n\ndef load_report_features(\n    cache_dir: str | Path,\n    sample_id: str,\n    expected_metadata: dict[str, Any] | None = None,\n) -> tuple[torch.Tensor, torch.Tensor, dict[str, Any]] | None:\n    cache_path = feature_cache_path(cache_dir, sample_id)\n    if not cache_path.exists():\n        return None\n    payload = torch.load(cache_path, map_location="cpu", weights_only=True)\n    metadata = payload.get("metadata", {})\n    if not _metadata_matches(metadata, expected_metadata):\n        return None\n    return payload["text"], payload["attributes"], metadata\n\n\nclass StructuredReportParser:\n    """Heuristic parser that extracts coarse structured report attributes."""\n\n    def __init__(self) -> None:\n        self.parser_name = "structured-report-parser-v1"\n        self.patterns = {\n            "laterality": {\n                "bilateral": re.compile(r"\\bbilateral\\b|\\bboth lungs?\\b", re.IGNORECASE),\n                "left": re.compile(r"\\bleft\\b", re.IGNORECASE),\n                "right": re.compile(r"\\bright\\b", re.IGNORECASE),\n                "diffuse": re.compile(r"\\bdiffuse\\b|\\bscattered\\b", re.IGNORECASE),\n            },\n            "vertical": {\n                "upper": re.compile(r"\\bupper\\b|\\bapical\\b", re.IGNORECASE),\n                "middle": re.compile(r"\\bmiddle\\b|\\bmid\\b", re.IGNORECASE),\n                "lower": re.compile(r"\\blower\\b|\\bbasal\\b|\\bbase\\b", re.IGNORECASE),\n                "basal": re.compile(r"\\bbasal\\b|\\bbase\\b", re.IGNORECASE),\n            },\n            "count": {\n                "single": re.compile(r"\\bsingle\\b|\\bone lesion\\b", re.IGNORECASE),\n                "multiple": re.compile(r"\\bmultiple\\b|\\bseveral\\b|\\bmultifocal\\b", re.IGNORECASE),\n                "diffuse": re.compile(r"\\bdiffuse\\b|\\bwidespread\\b", re.IGNORECASE),\n            },\n            "extent": {\n                "focal": re.compile(r"\\bfocal\\b|\\blocalized\\b", re.IGNORECASE),\n                "multifocal": re.compile(r"\\bmultifocal\\b|\\bpatchy\\b", re.IGNORECASE),\n                "diffuse": re.compile(r"\\bdiffuse\\b|\\bextensive\\b|\\bconfluent\\b", re.IGNORECASE),\n            },\n            "severity": {\n                "mild": re.compile(r"\\bmild\\b", re.IGNORECASE),\n                "moderate": re.compile(r"\\bmoderate\\b", re.IGNORECASE),\n                "severe": re.compile(r"\\bsevere\\b|\\bcritical\\b|\\bextensive\\b", re.IGNORECASE),\n            },\n        }\n\n    def parse(self, report: str) -> dict[str, str]:\n        text = report or ""\n        parsed: dict[str, str] = {}\n        for group, options in ATTRIBUTE_GROUPS.items():\n            parsed[group] = "unknown"\n            for option in options:\n                if option == "unknown":\n                    continue\n                pattern = self.patterns[group].get(option)\n                if pattern is not None and pattern.search(text):\n                    parsed[group] = option\n                    break\n        return parsed\n\n    def vectorize(self, report: str) -> torch.Tensor:\n        parsed = self.parse(report)\n        vector = torch.zeros(attribute_vector_size(), dtype=torch.float32)\n        offset = 0\n        for group, options in ATTRIBUTE_GROUPS.items():\n            value = parsed[group]\n            index = options.index(value)\n            vector[offset + index] = 1.0\n            offset += len(options)\n        return vector\n\n\nclass CachedDomainTextEncoder:\n    """Cached domain-aware text encoder for medical report text."""\n\n    def __init__(\n        self,\n        model_name: str = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",\n        device: str | None = None,\n        local_files_only: bool = False,\n    ) -> None:\n        self.model_name = model_name\n        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n        self.local_files_only = local_files_only\n        self._tokenizer = None\n        self._model = None\n        self._cache: dict[str, torch.Tensor] = {}\n        self._lock = threading.Lock()\n        self.report_parser = StructuredReportParser()\n\n    def cache_metadata(self, max_units: int) -> dict[str, Any]:\n        return build_cache_metadata(\n            model_name=self.model_name,\n            max_units=max_units,\n            parser_name=self.report_parser.parser_name,\n        )\n\n    def _ensure_model(self) -> None:\n        if self._model is not None and self._tokenizer is not None:\n            return\n        with self._lock:\n            if self._model is None or self._tokenizer is None:\n                from transformers import AutoModel, AutoTokenizer\n\n                try:\n                    self._tokenizer = AutoTokenizer.from_pretrained(\n                        self.model_name,\n                        local_files_only=self.local_files_only,\n                        use_fast=False,\n                    )\n                    self._model = AutoModel.from_pretrained(\n                        self.model_name,\n                        local_files_only=self.local_files_only,\n                        use_safetensors=True,\n                    )\n                except Exception as exc:\n                    raise RuntimeError(\n                        "Failed to initialize the Hugging Face text encoder. "\n                        "Make sure the requested model is available, the environment has "\n                        "compatible tokenizer dependencies, and offline mode has a cached copy. "\n                        "Recommended setup: transformers<5, protobuf, and sentencepiece installed."\n                    ) from exc\n                self._model.eval()\n                self._model.to(self.device)\n\n    def encode_lines(self, lines: Iterable[str], max_lines: int | None = None) -> torch.Tensor:\n        cleaned_lines: List[str] = [line.strip() for line in lines if line and line.strip()]\n        if max_lines is not None:\n            cleaned_lines = cleaned_lines[:max_lines]\n        if not cleaned_lines:\n            cleaned_lines = ["[NO_TEXT]"]\n        outputs = [self._encode_single(line) for line in cleaned_lines]\n        if max_lines is not None and len(outputs) < max_lines:\n            pad_token = outputs[0].new_zeros(outputs[0].shape)\n            outputs.extend([pad_token.clone() for _ in range(max_lines - len(outputs))])\n        return torch.stack(outputs, dim=0)\n\n    def encode_report(self, report: str, max_lines: int | None = None) -> tuple[torch.Tensor, torch.Tensor]:\n        lines = split_report_into_units(report)\n        tokens = self.encode_lines(lines, max_lines=max_lines)\n        attrs = self.report_parser.vectorize(report)\n        return tokens, attrs\n\n    def batch_attribute_vectors(self, reports: Sequence[str]) -> torch.Tensor:\n        return torch.stack([self.report_parser.vectorize(report) for report in reports], dim=0)\n\n    def _encode_single(self, text: str) -> torch.Tensor:\n        cached = self._cache.get(text)\n        if cached is not None:\n            return cached.clone()\n\n        self._ensure_model()\n        assert self._tokenizer is not None\n        assert self._model is not None\n        with torch.inference_mode():\n            encoded = self._tokenizer(\n                text,\n                truncation=True,\n                max_length=64,\n                padding="max_length",\n                return_tensors="pt",\n            )\n            encoded = {key: value.to(self.device) for key, value in encoded.items()}\n            outputs = self._model(**encoded)\n            embedding = outputs.last_hidden_state[:, 0, :].squeeze(0).detach().cpu()\n\n        self._cache[text] = embedding\n        return embedding.clone()\n\n\n# Backward-compatible alias for earlier local changes.\nCachedTextEncoder = CachedDomainTextEncoder\n'
LOAD_DATASET_SOURCE = '# -*- coding: utf-8 -*-\nimport os\nimport random\nfrom typing import Any, Callable\n\nimport cv2\nimport numpy as np\nimport torch\nfrom scipy import ndimage\nfrom torch.utils.data import Dataset\n\nfrom text_encoder import CachedTextEncoder, load_report_features, save_report_features\n\n\ndef random_rot_flip(image: np.ndarray, label: np.ndarray) -> tuple[np.ndarray, np.ndarray]:\n    k = np.random.randint(0, 4)\n    image = np.rot90(image, k, axes=(0, 1)).copy()\n    label = np.rot90(label, k, axes=(0, 1)).copy()\n    axis = np.random.randint(0, 2)\n    image = np.flip(image, axis=axis).copy()\n    label = np.flip(label, axis=axis).copy()\n    return image, label\n\n\ndef random_rotate(image: np.ndarray, label: np.ndarray) -> tuple[np.ndarray, np.ndarray]:\n    angle = np.random.randint(-20, 20)\n    image = ndimage.rotate(image, angle, axes=(0, 1), order=3, reshape=False, mode="nearest")\n    label = ndimage.rotate(label, angle, axes=(0, 1), order=0, reshape=False, mode="nearest")\n    return image, label\n\n\ndef _ensure_channel_last(array: np.ndarray) -> np.ndarray:\n    if array.ndim == 2:\n        return np.expand_dims(array, axis=-1)\n    return array\n\n\ndef _resize_image(image: np.ndarray, output_size: list[int] | tuple[int, int]) -> np.ndarray:\n    height, width = int(output_size[0]), int(output_size[1])\n    if image.shape[0] == height and image.shape[1] == width:\n        return image\n    resized = cv2.resize(image, (width, height), interpolation=cv2.INTER_LINEAR)\n    return _ensure_channel_last(resized)\n\n\ndef _resize_mask(mask: np.ndarray, output_size: list[int] | tuple[int, int]) -> np.ndarray:\n    height, width = int(output_size[0]), int(output_size[1])\n    if mask.shape[0] == height and mask.shape[1] == width:\n        return mask\n    resized = cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST)\n    return _ensure_channel_last(resized)\n\n\ndef to_long_tensor(mask: np.ndarray) -> torch.Tensor:\n    array = np.asarray(mask)\n    if array.ndim == 3 and array.shape[-1] == 1:\n        array = array[..., 0]\n    return torch.from_numpy(np.ascontiguousarray(array.astype(np.int64)))\n\n\ndef _to_image_tensor(image: np.ndarray) -> torch.Tensor:\n    array = np.asarray(image, dtype=np.float32)\n    if array.ndim == 2:\n        array = np.expand_dims(array, axis=-1)\n    if array.max() > 1.0:\n        array = array / 255.0\n    tensor = torch.from_numpy(np.ascontiguousarray(array.transpose(2, 0, 1)))\n    return tensor.float()\n\n\ndef _prepare_sample_tensor_dict(sample: dict[str, Any], output_size: list[int] | tuple[int, int]) -> dict[str, Any]:\n    image = sample.get("image")\n    label = sample["label"]\n    text = sample["text"]\n    attributes = sample.get("attributes")\n\n    if image is not None:\n        image = _ensure_channel_last(np.asarray(image))\n        image = _resize_image(image, output_size)\n    label = _ensure_channel_last(np.asarray(label))\n    label = _resize_mask(label, output_size)\n\n    prepared: dict[str, Any] = {\n        "label": to_long_tensor((label > 0).astype(np.uint8)),\n        "text": torch.as_tensor(text, dtype=torch.float32),\n    }\n    if image is not None:\n        prepared["image"] = _to_image_tensor(image)\n    if attributes is not None:\n        prepared["attributes"] = torch.as_tensor(attributes, dtype=torch.float32)\n    return prepared\n\n\nclass RandomGenerator(object):\n    def __init__(self, output_size):\n        self.output_size = output_size\n\n    def __call__(self, sample):\n        image = sample.get("image")\n        label = sample["label"]\n        if image is not None:\n            image = _ensure_channel_last(np.asarray(image))\n        label = _ensure_channel_last(np.asarray(label))\n\n        if image is not None and random.random() > 0.5:\n            image, label = random_rot_flip(image, label)\n        elif image is not None and random.random() > 0.5:\n            image, label = random_rotate(image, label)\n\n        transformed = dict(sample)\n        if image is not None:\n            transformed["image"] = image\n        transformed["label"] = label\n        return _prepare_sample_tensor_dict(transformed, self.output_size)\n\n\nclass ValGenerator(object):\n    def __init__(self, output_size):\n        self.output_size = output_size\n\n    def __call__(self, sample):\n        return _prepare_sample_tensor_dict(sample, self.output_size)\n\n\nclass _BaseTextDataset(Dataset):\n    def __init__(\n        self,\n        task_name: str,\n        row_text: dict[str, str],\n        cache_dir: str | None = None,\n        text_model_name: str | None = None,\n        local_files_only: bool = False,\n        cache_metadata: dict[str, Any] | None = None,\n        max_text_units: int = 10,\n    ) -> None:\n        self.task_name = task_name\n        self.rowtext = row_text\n        self.cache_dir = cache_dir\n        self.text_model_name = text_model_name\n        self.local_files_only = local_files_only\n        self.cache_metadata = cache_metadata\n        self.max_text_units = max_text_units\n        self.text_encoder: CachedTextEncoder | None = None\n\n        if cache_dir is None:\n            self.text_encoder = self._build_text_encoder()\n\n    def _build_text_encoder(self) -> CachedTextEncoder:\n        encoder = CachedTextEncoder(\n            model_name=self.text_model_name or "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",\n            local_files_only=self.local_files_only,\n        )\n        if self.cache_metadata is None:\n            self.cache_metadata = encoder.cache_metadata(self.max_text_units)\n        return encoder\n\n    def _get_text_encoder(self) -> CachedTextEncoder:\n        if self.text_encoder is None:\n            self.text_encoder = self._build_text_encoder()\n        return self.text_encoder\n\n    def _load_cached_features(self, *sample_ids: str):\n        if not self.cache_dir:\n            return None\n        for sample_id in sample_ids:\n            cached = load_report_features(\n                self.cache_dir,\n                sample_id,\n                expected_metadata=self.cache_metadata,\n            )\n            if cached is not None:\n                return cached\n        return None\n\n    def _get_text_features(self, report: str, *sample_ids: str) -> tuple[torch.Tensor, torch.Tensor]:\n        cached = self._load_cached_features(*sample_ids)\n        if cached is not None:\n            text, attributes, _ = cached\n            return text, attributes\n\n        encoder = self._get_text_encoder()\n        text, attributes = encoder.encode_report(report, max_lines=self.max_text_units)\n        if self.cache_dir:\n            metadata = self.cache_metadata or encoder.cache_metadata(self.max_text_units)\n            for sample_id in sample_ids:\n                save_report_features(self.cache_dir, sample_id, text, attributes, metadata=metadata)\n        return text, attributes\n\n\nclass LV2D(_BaseTextDataset):\n    def __init__(\n        self,\n        dataset_path: str,\n        task_name: str,\n        row_text: dict[str, str],\n        joint_transform: Callable = None,\n        one_hot_mask: int = False,\n        image_size: int = 224,\n        cache_dir: str | None = None,\n        text_model_name: str | None = None,\n        local_files_only: bool = False,\n        cache_metadata: dict[str, Any] | None = None,\n        max_text_units: int = 10,\n    ) -> None:\n        super().__init__(\n            task_name=task_name,\n            row_text=row_text,\n            cache_dir=cache_dir,\n            text_model_name=text_model_name,\n            local_files_only=local_files_only,\n            cache_metadata=cache_metadata,\n            max_text_units=max_text_units,\n        )\n        self.dataset_path = dataset_path\n        self.image_size = image_size\n        self.output_path = os.path.join(dataset_path)\n        self.mask_list = sorted(os.listdir(self.output_path))\n        self.one_hot_mask = one_hot_mask\n        self.joint_transform = joint_transform or ValGenerator(output_size=[image_size, image_size])\n\n    def __len__(self):\n        return len(self.mask_list)\n\n    def _resolve_report(self, mask_filename: str) -> str:\n        candidates = [mask_filename, os.path.splitext(mask_filename)[0]]\n        for candidate in candidates:\n            if candidate in self.rowtext:\n                return self.rowtext[candidate]\n        raise KeyError(f"Could not resolve text annotation for mask={mask_filename}")\n\n    def __getitem__(self, idx):\n        mask_filename = self.mask_list[idx]\n        mask = cv2.imread(os.path.join(self.output_path, mask_filename), cv2.IMREAD_GRAYSCALE)\n        if mask is None:\n            raise FileNotFoundError(f"Unable to read mask {mask_filename} from {self.output_path}")\n        mask = (mask > 0).astype(np.uint8)\n\n        report = self._resolve_report(mask_filename)\n        text, attributes = self._get_text_features(report, mask_filename, os.path.splitext(mask_filename)[0])\n        sample = {"label": mask, "text": text, "attributes": attributes}\n\n        if self.joint_transform:\n            sample = self.joint_transform(sample)\n\n        if self.one_hot_mask:\n            assert self.one_hot_mask > 0, "one_hot_mask must be nonnegative"\n            label = sample["label"]\n            sample["label"] = torch.zeros(\n                (self.one_hot_mask, label.shape[0], label.shape[1]),\n                dtype=torch.float32,\n            ).scatter_(0, label.unsqueeze(0), 1.0)\n\n        return sample, mask_filename\n\n\nclass ImageToImage2D(_BaseTextDataset):\n    def __init__(\n        self,\n        dataset_path: str,\n        task_name: str,\n        row_text: dict[str, str],\n        joint_transform: Callable = None,\n        one_hot_mask: int = False,\n        image_size: int = 224,\n        cache_dir: str | None = None,\n        text_model_name: str | None = None,\n        local_files_only: bool = False,\n        cache_metadata: dict[str, Any] | None = None,\n        max_text_units: int = 10,\n    ) -> None:\n        super().__init__(\n            task_name=task_name,\n            row_text=row_text,\n            cache_dir=cache_dir,\n            text_model_name=text_model_name,\n            local_files_only=local_files_only,\n            cache_metadata=cache_metadata,\n            max_text_units=max_text_units,\n        )\n        self.dataset_path = dataset_path\n        self.image_size = image_size\n        self.input_path = os.path.join(dataset_path, "img")\n        self.output_path = os.path.join(dataset_path, "labelcol")\n        self.images_list = sorted(os.listdir(self.input_path))\n        self.mask_list = sorted(os.listdir(self.output_path))\n        self.one_hot_mask = one_hot_mask\n        self.joint_transform = joint_transform or ValGenerator(output_size=[image_size, image_size])\n\n    def __len__(self):\n        return len(self.images_list)\n\n    def _resolve_mask_filename(self, image_filename: str) -> str:\n        image_stem = os.path.splitext(image_filename)[0]\n        candidates = [\n            f"{image_stem}.png",\n            f"{image_stem}.jpg",\n            f"{image_stem}.jpeg",\n            image_filename,\n            image_filename.replace("mask_", ""),\n            image_filename.replace(".tif", ".png"),\n            image_filename.replace(".tiff", ".png"),\n        ]\n        for candidate in candidates:\n            if candidate in self.mask_list:\n                return candidate\n        raise FileNotFoundError(f"Could not resolve mask for image {image_filename} in {self.output_path}")\n\n    def _resolve_report(self, image_filename: str, mask_filename: str) -> str:\n        candidates = [\n            mask_filename,\n            image_filename,\n            os.path.splitext(mask_filename)[0],\n            os.path.splitext(image_filename)[0],\n        ]\n        for candidate in candidates:\n            if candidate in self.rowtext:\n                return self.rowtext[candidate]\n        raise KeyError(f"Could not resolve text annotation for image={image_filename}, mask={mask_filename}")\n\n    def __getitem__(self, idx):\n        image_filename = self.images_list[idx]\n        mask_filename = self._resolve_mask_filename(image_filename)\n\n        image = cv2.imread(os.path.join(self.input_path, image_filename), cv2.IMREAD_COLOR)\n        if image is None:\n            raise FileNotFoundError(f"Unable to read image {image_filename} from {self.input_path}")\n        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)\n\n        mask = cv2.imread(os.path.join(self.output_path, mask_filename), cv2.IMREAD_GRAYSCALE)\n        if mask is None:\n            raise FileNotFoundError(f"Unable to read mask {mask_filename} from {self.output_path}")\n        mask = (mask > 0).astype(np.uint8)\n\n        report = self._resolve_report(image_filename, mask_filename)\n        text, attributes = self._get_text_features(\n            report,\n            mask_filename,\n            image_filename,\n            os.path.splitext(mask_filename)[0],\n            os.path.splitext(image_filename)[0],\n        )\n        sample = {"image": image, "label": mask, "text": text, "attributes": attributes}\n\n        if self.joint_transform:\n            sample = self.joint_transform(sample)\n\n        if self.one_hot_mask:\n            assert self.one_hot_mask > 0, "one_hot_mask must be nonnegative"\n            label = sample["label"]\n            sample["label"] = torch.zeros(\n                (self.one_hot_mask, label.shape[0], label.shape[1]),\n                dtype=torch.float32,\n            ).scatter_(0, label.unsqueeze(0), 1.0)\n\n        return sample, image_filename\n'
IMPROVED_SSL_SOURCE = 'import math\nfrom typing import Optional, Tuple\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nclass BottleneckCrossAttention(nn.Module):\n    """True image-text interaction at the semantic bottleneck."""\n\n    def __init__(self, dim: int, num_heads: int = 8, num_bottleneck_tokens: int = 4) -> None:\n        super().__init__()\n        self.dim = dim\n        self.num_bottleneck_tokens = num_bottleneck_tokens\n        self.bottleneck_tokens = nn.Parameter(torch.randn(num_bottleneck_tokens, dim) * 0.02)\n        self.text_to_bottleneck = nn.MultiheadAttention(dim, num_heads, batch_first=True)\n        self.vision_to_bottleneck = nn.MultiheadAttention(dim, num_heads, batch_first=True)\n        self.bottleneck_to_vision = nn.MultiheadAttention(dim, num_heads, batch_first=True)\n        self.gate = nn.Sequential(\n            nn.LayerNorm(dim * 2),\n            nn.Linear(dim * 2, dim),\n            nn.GELU(),\n            nn.Linear(dim, dim),\n            nn.Sigmoid(),\n        )\n        self.output_norm = nn.LayerNorm(dim)\n\n    def forward(\n        self,\n        visual_tokens: torch.Tensor,\n        text_tokens: Optional[torch.Tensor] = None,\n    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        batch_size = visual_tokens.shape[0]\n        bottleneck = self.bottleneck_tokens.unsqueeze(0).expand(batch_size, -1, -1)\n\n        if text_tokens is None:\n            text_tokens = visual_tokens.new_zeros(batch_size, 1, self.dim)\n\n        bottleneck = bottleneck + self.text_to_bottleneck(bottleneck, text_tokens, text_tokens)[0]\n        bottleneck = bottleneck + self.vision_to_bottleneck(bottleneck, visual_tokens, visual_tokens)[0]\n\n        pooled_visual = visual_tokens.mean(dim=1)\n        pooled_text = text_tokens.mean(dim=1)\n        gate = self.gate(torch.cat([pooled_visual, pooled_text], dim=-1)).unsqueeze(1)\n\n        attended_visual = self.bottleneck_to_vision(visual_tokens, bottleneck, bottleneck)[0]\n        fused_visual = self.output_norm(visual_tokens + gate * attended_visual)\n        return fused_visual, bottleneck, gate\n\n\nclass StructuredTextConditioner(nn.Module):\n    """Projects text tokens and structured attributes into the fusion space."""\n\n    def __init__(\n        self,\n        text_dim: int,\n        model_dim: int,\n        attribute_dim: int = 0,\n        text_dropout_prob: float = 0.3,\n    ) -> None:\n        super().__init__()\n        self.model_dim = model_dim\n        self.attribute_dim = attribute_dim\n        self.text_dropout_prob = text_dropout_prob\n        self.token_projection = nn.Sequential(\n            nn.LayerNorm(text_dim),\n            nn.Linear(text_dim, model_dim),\n        )\n        self.attribute_projection = (\n            nn.Sequential(\n                nn.LayerNorm(attribute_dim),\n                nn.Linear(attribute_dim, model_dim),\n                nn.GELU(),\n            )\n            if attribute_dim > 0\n            else None\n        )\n        self.no_text_token = nn.Parameter(torch.randn(1, 1, model_dim) * 0.02)\n\n    def forward(\n        self,\n        text_tokens: Optional[torch.Tensor],\n        structured_attributes: Optional[torch.Tensor],\n        prototype_summary: torch.Tensor,\n        force_drop_mask: Optional[torch.Tensor] = None,\n    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:\n        batch_size = prototype_summary.shape[0]\n        device = prototype_summary.device\n        dtype = prototype_summary.dtype\n\n        if self.attribute_projection is not None and structured_attributes is not None:\n            structured_attributes = structured_attributes.to(device=device, dtype=dtype)\n            attribute_context = self.attribute_projection(structured_attributes)\n        else:\n            attribute_context = prototype_summary.new_zeros(batch_size, self.model_dim)\n\n        if text_tokens is not None:\n            projected_text = self.token_projection(text_tokens.float())\n            projected_text = projected_text.to(device=device)\n            projected_text = projected_text + attribute_context.unsqueeze(1)\n            fallback_mask = torch.zeros(batch_size, device=device, dtype=torch.bool)\n            if self.training and self.text_dropout_prob > 0:\n                fallback_mask = torch.rand(batch_size, device=device) < self.text_dropout_prob\n            if force_drop_mask is not None:\n                fallback_mask = fallback_mask | force_drop_mask.to(device=device, dtype=torch.bool)\n        else:\n            projected_text = None\n            fallback_mask = torch.ones(batch_size, device=device, dtype=torch.bool)\n\n        fallback_tokens = self.no_text_token.expand(batch_size, -1, -1) + attribute_context.unsqueeze(1)\n        text_summary = prototype_summary + attribute_context\n\n        if projected_text is None:\n            fallback_tokens = fallback_tokens.to(device=device, dtype=prototype_summary.dtype)\n            prepared_tokens = fallback_tokens\n        else:\n            fallback_tokens = fallback_tokens.to(device=projected_text.device, dtype=projected_text.dtype)\n            prepared_tokens = projected_text.clone()\n            expanded_fallback = fallback_tokens.expand(-1, projected_text.shape[1], -1)\n            text_summary = projected_text.mean(dim=1).to(device=device, dtype=prototype_summary.dtype)\n            prepared_tokens[fallback_mask] = expanded_fallback[fallback_mask]\n            text_summary[fallback_mask] = prototype_summary[fallback_mask] + attribute_context[fallback_mask]\n\n        return prepared_tokens, text_summary, attribute_context, fallback_mask\n\n\nclass TextSpatialPrior(nn.Module):\n    """Converts a text summary into a coarse probabilistic spatial prior map."""\n\n    def __init__(self, dim: int, height: int, width: int, num_basis: int = 7, attribute_dim: int = 0) -> None:\n        super().__init__()\n        self.height = height\n        self.width = width\n        self.num_basis = num_basis\n        self.attribute_dim = attribute_dim\n        self.register_buffer("basis_maps", self._build_basis_maps(height, width, num_basis), persistent=False)\n        self.summary_to_basis = nn.Sequential(\n            nn.LayerNorm(dim),\n            nn.Linear(dim, dim),\n            nn.GELU(),\n            nn.Linear(dim, num_basis),\n        )\n        self.attribute_to_basis = nn.Linear(attribute_dim, num_basis) if attribute_dim > 0 else None\n        self.refine = nn.Sequential(\n            nn.Conv2d(1, 8, kernel_size=3, padding=1),\n            nn.GELU(),\n            nn.Conv2d(8, 1, kernel_size=1),\n        )\n\n    @staticmethod\n    def _build_basis_maps(height: int, width: int, num_basis: int) -> torch.Tensor:\n        y = torch.linspace(-1.0, 1.0, height)\n        x = torch.linspace(-1.0, 1.0, width)\n        yy, xx = torch.meshgrid(y, x, indexing="ij")\n\n        center = torch.exp(-(xx.square() + yy.square()) / 0.5)\n        left = torch.clamp(-xx, min=0.0)\n        right = torch.clamp(xx, min=0.0)\n        upper = torch.clamp(-yy, min=0.0)\n        lower = torch.clamp(yy, min=0.0)\n        vertical_band = torch.exp(-xx.square() / 0.25)\n        horizontal_band = torch.exp(-yy.square() / 0.25)\n\n        basis = torch.stack(\n            [center, left, right, upper, lower, vertical_band, horizontal_band],\n            dim=0,\n        )\n        if num_basis < basis.shape[0]:\n            basis = basis[:num_basis]\n        elif num_basis > basis.shape[0]:\n            repeats = math.ceil(num_basis / basis.shape[0])\n            basis = basis.repeat(repeats, 1, 1)[:num_basis]\n        basis = basis / basis.amax(dim=(1, 2), keepdim=True).clamp_min(1e-6)\n        return basis\n\n    def forward(\n        self,\n        text_tokens: Optional[torch.Tensor] = None,\n        fallback_summary: Optional[torch.Tensor] = None,\n        structured_attributes: Optional[torch.Tensor] = None,\n    ) -> torch.Tensor:\n        if text_tokens is None:\n            if fallback_summary is None:\n                batch_size = 1\n                device = self.basis_maps.device\n                dtype = self.basis_maps.dtype\n                return torch.zeros(batch_size, 1, self.height, self.width, device=device, dtype=dtype)\n            summary = fallback_summary\n        else:\n            summary = text_tokens.mean(dim=1)\n\n        logits = self.summary_to_basis(summary)\n        if self.attribute_to_basis is not None and structured_attributes is not None:\n            logits = logits + self.attribute_to_basis(structured_attributes.to(device=summary.device, dtype=summary.dtype))\n        weights = torch.softmax(logits, dim=-1)\n        prior = torch.einsum("bk,khw->bhw", weights, self.basis_maps)\n        prior = prior.unsqueeze(1)\n        return torch.sigmoid(self.refine(prior) + prior)\n\n\nclass PrototypeMemoryBank(nn.Module):\n    """Stores semantic prototypes so the model can fall back when text is missing."""\n\n    def __init__(self, dim: int, num_prototypes: int = 8, momentum: float = 0.1, temperature: float = 0.1) -> None:\n        super().__init__()\n        memory = F.normalize(torch.randn(num_prototypes, dim), dim=-1)\n        self.register_buffer("memory", memory)\n        self.momentum = momentum\n        self.temperature = temperature\n\n    def retrieve(self, query: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:\n        normalized_query = F.normalize(query, dim=-1)\n        normalized_memory = F.normalize(self.memory, dim=-1)\n        logits = normalized_query @ normalized_memory.transpose(0, 1) / self.temperature\n        weights = torch.softmax(logits, dim=-1)\n        retrieved = weights @ self.memory\n        return retrieved, weights\n\n    @torch.no_grad()\n    def update(self, features: torch.Tensor, valid_mask: Optional[torch.Tensor] = None) -> None:\n        features = features.to(device=self.memory.device, dtype=self.memory.dtype)\n        if valid_mask is not None:\n            valid_mask = valid_mask.to(device=features.device, dtype=torch.bool)\n            if not valid_mask.any():\n                return\n            features = features[valid_mask]\n        normalized_features = F.normalize(features, dim=-1)\n        _, weights = self.retrieve(normalized_features)\n        assignment_mass = weights.sum(dim=0, keepdim=True).transpose(0, 1)\n        aggregated = weights.transpose(0, 1) @ normalized_features\n        valid = assignment_mass.squeeze(-1) > 0\n        if valid.any():\n            aggregated[valid] = aggregated[valid] / assignment_mass[valid].clamp_min(1e-6)\n            updated = self.memory.clone()\n            updated[valid] = F.normalize(\n                (1.0 - self.momentum) * updated[valid] + self.momentum * aggregated[valid],\n                dim=-1,\n            )\n            self.memory.copy_(updated)\n\n\nclass CrossModalSkipAdapter(nn.Module):\n    """Modulates decoder skip features using fused bottleneck context and the text prior."""\n\n    def __init__(self, skip_channels: int, context_dim: int) -> None:\n        super().__init__()\n        self.channel_gate = nn.Sequential(\n            nn.LayerNorm(context_dim),\n            nn.Linear(context_dim, skip_channels),\n            nn.GELU(),\n            nn.Linear(skip_channels, skip_channels),\n        )\n        self.spatial_gate = nn.Conv2d(1, skip_channels, kernel_size=1)\n        self.refine = nn.Sequential(\n            nn.Conv2d(skip_channels, skip_channels, kernel_size=3, padding=1),\n            nn.BatchNorm2d(skip_channels),\n            nn.GELU(),\n        )\n\n    def forward(self, skip: torch.Tensor, context: torch.Tensor, spatial_prior: torch.Tensor) -> torch.Tensor:\n        channel = torch.sigmoid(self.channel_gate(context)).unsqueeze(-1).unsqueeze(-1)\n        prior = F.interpolate(spatial_prior, size=skip.shape[-2:], mode="bilinear", align_corners=False)\n        spatial = torch.sigmoid(self.spatial_gate(prior))\n        adapted = skip * (1.0 + channel) * (1.0 + spatial)\n        return self.refine(adapted)\n\n\nclass UncertaintyAwarePseudoLabelFusion(nn.Module):\n    """Merges visual pseudo-labels and text priors only when they are reliable and agreeing."""\n\n    def __init__(\n        self,\n        entropy_threshold: float = 0.35,\n        agreement_threshold: float = 0.5,\n        coarse_size: int = 16,\n    ) -> None:\n        super().__init__()\n        self.entropy_threshold = entropy_threshold\n        self.agreement_threshold = agreement_threshold\n        self.coarse_size = coarse_size\n\n    @staticmethod\n    def _binary_entropy(probs: torch.Tensor) -> torch.Tensor:\n        probs = probs.clamp(1e-6, 1.0 - 1e-6)\n        return -(probs * probs.log() + (1.0 - probs) * (1.0 - probs).log())\n\n    @staticmethod\n    def _dice_score(lhs: torch.Tensor, rhs: torch.Tensor) -> torch.Tensor:\n        lhs = lhs.reshape(lhs.shape[0], -1)\n        rhs = rhs.reshape(rhs.shape[0], -1)\n        intersection = (lhs * rhs).sum(dim=1)\n        union = lhs.sum(dim=1) + rhs.sum(dim=1)\n        return (2.0 * intersection + 1e-6) / (union + 1e-6)\n\n    def _coarse_binary(self, probs: torch.Tensor) -> torch.Tensor:\n        pooled = F.adaptive_avg_pool2d(probs, (self.coarse_size, self.coarse_size))\n        return (pooled > 0.5).float()\n\n    def forward(self, visual_probs: torch.Tensor, text_prior_probs: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:\n        entropy = self._binary_entropy(visual_probs).mean(dim=(1, 2, 3))\n        confident = entropy < self.entropy_threshold\n        agreement = self._dice_score(self._coarse_binary(visual_probs), self._coarse_binary(text_prior_probs))\n        agreeing = agreement > self.agreement_threshold\n        weight = (confident & agreeing).float().view(-1, 1, 1, 1)\n        fused = weight * visual_probs + (1.0 - weight) * text_prior_probs.detach()\n        return fused, weight\n\n\nclass BoundaryConsistencyLoss(nn.Module):\n    """Encourages boundaries of pseudo-labels and predictions to agree."""\n\n    def __init__(self) -> None:\n        super().__init__()\n        sobel_x = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]])\n        sobel_y = sobel_x.transpose(0, 1)\n        self.register_buffer("sobel_x", sobel_x.view(1, 1, 3, 3), persistent=False)\n        self.register_buffer("sobel_y", sobel_y.view(1, 1, 3, 3), persistent=False)\n\n    def _edge_map(self, tensor: torch.Tensor) -> torch.Tensor:\n        sobel_x = self.sobel_x.to(device=tensor.device, dtype=tensor.dtype)\n        sobel_y = self.sobel_y.to(device=tensor.device, dtype=tensor.dtype)\n        grad_x = F.conv2d(tensor, sobel_x, padding=1)\n        grad_y = F.conv2d(tensor, sobel_y, padding=1)\n        return torch.sqrt(grad_x.square() + grad_y.square() + 1e-6)\n\n    def forward(self, prediction: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        prediction = prediction.float()\n        target = target.float()\n        pred_edges = self._edge_map(prediction)\n        target_edges = self._edge_map(target)\n        return F.l1_loss(pred_edges, target_edges)\n\n\nclass PrototypeAlignmentLoss(nn.Module):\n    """Aligns visual summaries to the prototype-backed summary used for text-free inference."""\n\n    def forward(self, query: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        return 1.0 - F.cosine_similarity(query, target, dim=-1).mean()\n'
IMPROVED_MODEL_SOURCE = 'from typing import Dict, Optional, Tuple\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom .improved_ssl import (\n    BottleneckCrossAttention,\n    CrossModalSkipAdapter,\n    PrototypeMemoryBank,\n    StructuredTextConditioner,\n    TextSpatialPrior,\n)\n\n\nclass ConvBlock(nn.Module):\n    def __init__(self, in_channels: int, out_channels: int) -> None:\n        super().__init__()\n        self.block = nn.Sequential(\n            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),\n            nn.BatchNorm2d(out_channels),\n            nn.GELU(),\n            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),\n            nn.BatchNorm2d(out_channels),\n            nn.GELU(),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.block(x)\n\n\nclass DownsampleBlock(nn.Module):\n    def __init__(self, in_channels: int, out_channels: int) -> None:\n        super().__init__()\n        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)\n        self.block = ConvBlock(in_channels, out_channels)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.block(self.pool(x))\n\n\nclass UpsampleFusionBlock(nn.Module):\n    def __init__(self, in_channels: int, skip_channels: int, out_channels: int, context_dim: int) -> None:\n        super().__init__()\n        self.skip_adapter = CrossModalSkipAdapter(skip_channels, context_dim)\n        self.fuse = ConvBlock(in_channels + skip_channels, out_channels)\n\n    def forward(\n        self,\n        x: torch.Tensor,\n        skip: torch.Tensor,\n        context: torch.Tensor,\n        spatial_prior: torch.Tensor,\n    ) -> torch.Tensor:\n        upsampled = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)\n        adapted_skip = self.skip_adapter(skip, context, spatial_prior)\n        return self.fuse(torch.cat([upsampled, adapted_skip], dim=1))\n\n\nclass LViTImproved(nn.Module):\n    """\n    Improved LViT variant guided by the research report:\n    - true bottleneck image-text cross-attention\n    - explicit text-view spatial prior\n    - prototype fallback when text is missing\n    - cross-modal skip refinement during decoding\n    """\n\n    def __init__(\n        self,\n        n_channels: int = 3,\n        n_classes: int = 1,\n        img_size: int = 224,\n        text_dim: int = 768,\n        base_channels: int = 64,\n        bottleneck_dim: int = 512,\n        num_bottleneck_tokens: int = 4,\n        num_heads: int = 8,\n        transformer_layers: int = 2,\n        num_prototypes: int = 8,\n        prototype_momentum: float = 0.1,\n        attribute_dim: int = 0,\n        text_dropout_prob: float = 0.3,\n        enable_upper_scale_fusion: bool = True,\n    ) -> None:\n        super().__init__()\n        self.n_classes = n_classes\n        self.text_dim = text_dim\n        self.bottleneck_dim = bottleneck_dim\n        self.spatial_size = img_size // 16\n        self.attribute_dim = attribute_dim\n        self.enable_upper_scale_fusion = enable_upper_scale_fusion\n\n        self.stem = ConvBlock(n_channels, base_channels)\n        self.down1 = DownsampleBlock(base_channels, base_channels * 2)\n        self.down2 = DownsampleBlock(base_channels * 2, base_channels * 4)\n        self.down3 = DownsampleBlock(base_channels * 4, bottleneck_dim)\n        self.down4 = DownsampleBlock(bottleneck_dim, bottleneck_dim)\n\n        encoder_layer = nn.TransformerEncoderLayer(\n            d_model=bottleneck_dim,\n            nhead=num_heads,\n            dim_feedforward=bottleneck_dim * 4,\n            dropout=0.1,\n            batch_first=True,\n            activation="gelu",\n        )\n        self.visual_bottleneck = nn.TransformerEncoder(encoder_layer, num_layers=transformer_layers)\n        self.upper_scale_encoder = (\n            nn.TransformerEncoder(encoder_layer, num_layers=1) if enable_upper_scale_fusion else None\n        )\n        self.text_conditioner = StructuredTextConditioner(\n            text_dim=text_dim,\n            model_dim=bottleneck_dim,\n            attribute_dim=attribute_dim,\n            text_dropout_prob=text_dropout_prob,\n        )\n        self.prototype_memory = PrototypeMemoryBank(\n            dim=bottleneck_dim,\n            num_prototypes=num_prototypes,\n            momentum=prototype_momentum,\n        )\n        self.cross_attention = BottleneckCrossAttention(\n            dim=bottleneck_dim,\n            num_heads=num_heads,\n            num_bottleneck_tokens=num_bottleneck_tokens,\n        )\n        self.upper_scale_cross_attention = (\n            BottleneckCrossAttention(\n                dim=bottleneck_dim,\n                num_heads=num_heads,\n                num_bottleneck_tokens=num_bottleneck_tokens,\n            )\n            if enable_upper_scale_fusion\n            else None\n        )\n        self.text_prior = TextSpatialPrior(\n            dim=bottleneck_dim,\n            height=self.spatial_size,\n            width=self.spatial_size,\n            attribute_dim=attribute_dim,\n        )\n        self.prior_to_logits = nn.Conv2d(1, n_classes, kernel_size=1)\n\n        self.up4 = UpsampleFusionBlock(bottleneck_dim, bottleneck_dim, base_channels * 4, bottleneck_dim)\n        self.up3 = UpsampleFusionBlock(base_channels * 4, base_channels * 4, base_channels * 2, bottleneck_dim)\n        self.up2 = UpsampleFusionBlock(base_channels * 2, base_channels * 2, base_channels, bottleneck_dim)\n        self.up1 = UpsampleFusionBlock(base_channels, base_channels, base_channels, bottleneck_dim)\n\n        self.segmentation_head = nn.Conv2d(base_channels, n_classes, kernel_size=1)\n        self.output_bias = nn.Parameter(torch.tensor(0.0))\n\n    def _prepare_text(\n        self,\n        text_tokens: Optional[torch.Tensor],\n        structured_attributes: Optional[torch.Tensor],\n        visual_summary: torch.Tensor,\n        force_text_dropout_mask: Optional[torch.Tensor] = None,\n    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:\n        retrieved_summary, prototype_weights = self.prototype_memory.retrieve(visual_summary)\n        prepared_tokens, text_summary, attribute_context, fallback_mask = self.text_conditioner(\n            text_tokens=text_tokens,\n            structured_attributes=structured_attributes,\n            prototype_summary=retrieved_summary,\n            force_drop_mask=force_text_dropout_mask,\n        )\n        return prepared_tokens, text_summary, retrieved_summary, prototype_weights, fallback_mask, attribute_context\n\n    def forward(\n        self,\n        image: torch.Tensor,\n        text_tokens: Optional[torch.Tensor] = None,\n        structured_attributes: Optional[torch.Tensor] = None,\n        return_aux: bool = False,\n        update_prototypes: bool = False,\n        force_text_dropout_mask: Optional[torch.Tensor] = None,\n    ) -> torch.Tensor | Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        x1 = self.stem(image.float())\n        x2 = self.down1(x1)\n        x3 = self.down2(x2)\n        x4 = self.down3(x3)\n        x5 = self.down4(x4)\n\n        visual_tokens = x5.flatten(2).transpose(1, 2)\n        visual_tokens = self.visual_bottleneck(visual_tokens)\n        visual_summary = visual_tokens.mean(dim=1)\n\n        text_tokens_prepared, text_summary, prototype_summary, prototype_weights, used_prototype_fallback, attribute_context = (\n            self._prepare_text(\n                text_tokens,\n                structured_attributes,\n                visual_summary,\n                force_text_dropout_mask=force_text_dropout_mask,\n            )\n        )\n        fused_tokens, bottleneck_tokens, gate = self.cross_attention(visual_tokens, text_tokens_prepared)\n        fused_summary = fused_tokens.mean(dim=1)\n\n        upper_gate = None\n        if self.enable_upper_scale_fusion and self.upper_scale_cross_attention is not None and self.upper_scale_encoder is not None:\n            upper_tokens = x4.flatten(2).transpose(1, 2)\n            upper_tokens = self.upper_scale_encoder(upper_tokens)\n            upper_tokens, _, upper_gate = self.upper_scale_cross_attention(upper_tokens, text_tokens_prepared)\n            x4 = upper_tokens.transpose(1, 2).reshape_as(x4)\n\n        if self.training and update_prototypes and text_tokens is not None:\n            paired_mask = ~used_prototype_fallback\n            self.prototype_memory.update(fused_summary.detach(), valid_mask=paired_mask)\n\n        spatial_prior = self.text_prior(\n            text_tokens=text_tokens_prepared if text_tokens is not None else None,\n            fallback_summary=prototype_summary,\n            structured_attributes=structured_attributes,\n        )\n        bottleneck_map = fused_tokens.transpose(1, 2).reshape(\n            x5.shape[0],\n            self.bottleneck_dim,\n            self.spatial_size,\n            self.spatial_size,\n        )\n\n        y4 = self.up4(bottleneck_map, x4, fused_summary, spatial_prior)\n        y3 = self.up3(y4, x3, fused_summary, spatial_prior)\n        y2 = self.up2(y3, x2, fused_summary, spatial_prior)\n        y1 = self.up1(y2, x1, fused_summary, spatial_prior)\n\n        logits = self.segmentation_head(y1)\n        prior_logits = F.interpolate(\n            self.prior_to_logits(spatial_prior),\n            size=logits.shape[-2:],\n            mode="bilinear",\n            align_corners=False,\n        )\n        combined_logits = logits + prior_logits + self.output_bias\n        prior_prediction = torch.sigmoid(prior_logits)\n        probabilities = torch.sigmoid(combined_logits)\n\n        if not return_aux:\n            return combined_logits\n\n        aux = {\n            "spatial_prior": spatial_prior,\n            "prior_logits": prior_logits,\n            "prior_prediction": prior_prediction,\n            "probabilities": probabilities,\n            "bottleneck_tokens": bottleneck_tokens,\n            "fusion_gate": gate,\n            "upper_scale_gate": upper_gate,\n            "visual_summary": visual_summary,\n            "fused_summary": fused_summary,\n            "text_summary": text_summary,\n            "prototype_summary": prototype_summary,\n            "prototype_weights": prototype_weights,\n            "attribute_context": attribute_context,\n            "used_prototype_fallback": used_prototype_fallback.to(device=combined_logits.device, dtype=combined_logits.dtype),\n        }\n        return combined_logits, aux\n\n\ndef build_lvit_improved(**kwargs) -> LViTImproved:\n    return LViTImproved(**kwargs)\n'
IMPROVED_TRAINING_SOURCE = 'from contextlib import nullcontext\nfrom typing import Dict, Optional\n\nimport torch\nimport torch.nn as nn\n\nfrom .improved_ssl import BoundaryConsistencyLoss, PrototypeAlignmentLoss, UncertaintyAwarePseudoLabelFusion\n\n\nclass SoftDiceLoss(nn.Module):\n    def __init__(self, smooth: float = 1e-6) -> None:\n        super().__init__()\n        self.smooth = smooth\n\n    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        if target.ndim == logits.ndim - 1:\n            target = target.unsqueeze(1)\n        prediction = torch.sigmoid(logits).float().reshape(logits.shape[0], -1)\n        target = target.float().reshape(target.shape[0], -1)\n        intersection = (prediction * target).sum(dim=1)\n        union = prediction.sum(dim=1) + target.sum(dim=1)\n        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)\n        return 1.0 - dice.mean()\n\n\nclass ImprovedSemiSupervisedLoss(nn.Module):\n    """Composite objective matching the improved report-driven design."""\n\n    def __init__(\n        self,\n        lambda_consistency: float = 1.0,\n        lambda_text: float = 0.5,\n        lambda_boundary: float = 0.25,\n        lambda_prototype: float = 0.1,\n        entropy_threshold: float = 0.35,\n        agreement_threshold: float = 0.5,\n    ) -> None:\n        super().__init__()\n        self.supervised_bce = nn.BCEWithLogitsLoss()\n        self.supervised_dice = SoftDiceLoss()\n        self.consistency_bce = nn.BCEWithLogitsLoss()\n        self.fusion = UncertaintyAwarePseudoLabelFusion(\n            entropy_threshold=entropy_threshold,\n            agreement_threshold=agreement_threshold,\n        )\n        self.boundary = BoundaryConsistencyLoss()\n        self.prototype_alignment = PrototypeAlignmentLoss()\n        self.lambda_consistency = lambda_consistency\n        self.lambda_text = lambda_text\n        self.lambda_boundary = lambda_boundary\n        self.lambda_prototype = lambda_prototype\n\n    def forward(\n        self,\n        labeled_logits: torch.Tensor,\n        labeled_target: torch.Tensor,\n        unlabeled_logits: Optional[torch.Tensor] = None,\n        visual_teacher_prediction: Optional[torch.Tensor] = None,\n        text_prior_prediction: Optional[torch.Tensor] = None,\n        visual_summary: Optional[torch.Tensor] = None,\n        prototype_summary: Optional[torch.Tensor] = None,\n    ) -> Dict[str, torch.Tensor]:\n        if labeled_target.ndim == labeled_logits.ndim - 1:\n            labeled_target = labeled_target.unsqueeze(1)\n\n        if unlabeled_logits is not None and visual_teacher_prediction is not None:\n            if visual_teacher_prediction.ndim == unlabeled_logits.ndim - 1:\n                visual_teacher_prediction = visual_teacher_prediction.unsqueeze(1)\n\n        if unlabeled_logits is not None and text_prior_prediction is not None:\n            if text_prior_prediction.ndim == unlabeled_logits.ndim - 1:\n                text_prior_prediction = text_prior_prediction.unsqueeze(1)\n\n        supervised = self.supervised_bce(labeled_logits, labeled_target.float()) + self.supervised_dice(\n            labeled_logits,\n            labeled_target.float(),\n        )\n\n        consistency = supervised.new_tensor(0.0)\n        text_guidance = supervised.new_tensor(0.0)\n        boundary = supervised.new_tensor(0.0)\n        prototype = supervised.new_tensor(0.0)\n        fused_pseudo = None\n        fusion_weight = supervised.new_tensor(0.0)\n\n        if (\n            unlabeled_logits is not None\n            and visual_teacher_prediction is not None\n            and text_prior_prediction is not None\n        ):\n            fused_pseudo, weight = self.fusion(visual_teacher_prediction, text_prior_prediction)\n            consistency = self.consistency_bce(unlabeled_logits, fused_pseudo.detach())\n            text_guidance = self.consistency_bce(unlabeled_logits, text_prior_prediction.detach())\n            boundary = self.boundary(torch.sigmoid(unlabeled_logits), fused_pseudo.detach())\n            fusion_weight = weight.mean()\n\n        if visual_summary is not None and prototype_summary is not None:\n            prototype = self.prototype_alignment(visual_summary, prototype_summary.detach())\n\n        total = (\n            supervised\n            + self.lambda_consistency * consistency\n            + self.lambda_text * text_guidance\n            + self.lambda_boundary * boundary\n            + self.lambda_prototype * prototype\n        )\n\n        return {\n            "total": total,\n            "supervised": supervised,\n            "consistency": consistency,\n            "text_guidance": text_guidance,\n            "boundary": boundary,\n            "prototype": prototype,\n            "fusion_weight": fusion_weight,\n            "fused_pseudo": fused_pseudo if fused_pseudo is not None else torch.sigmoid(labeled_logits.detach()),\n        }\n\n\n@torch.no_grad()\ndef update_ema_model(teacher: nn.Module, student: nn.Module, decay: float = 0.99) -> None:\n    for teacher_param, student_param in zip(teacher.parameters(), student.parameters()):\n        teacher_param.data.mul_(decay).add_(student_param.data, alpha=1.0 - decay)\n\n    for teacher_buffer, student_buffer in zip(teacher.buffers(), student.buffers()):\n        teacher_buffer.copy_(student_buffer)\n\n\nclass ImprovedSSLTrainer:\n    """Minimal SSL-native trainer that wires the improved model into a dual-teacher loop."""\n\n    def __init__(\n        self,\n        student: nn.Module,\n        optimizer: torch.optim.Optimizer,\n        teacher: Optional[nn.Module] = None,\n        loss_fn: Optional[ImprovedSemiSupervisedLoss] = None,\n        ema_decay: float = 0.99,\n        student_view_noise_std: float = 0.05,\n        student_view_dropout_prob: float = 0.0,\n    ) -> None:\n        self.student = student\n        self.teacher = teacher if teacher is not None else self._clone_teacher(student)\n        self.optimizer = optimizer\n        self.loss_fn = loss_fn if loss_fn is not None else ImprovedSemiSupervisedLoss()\n        self.ema_decay = ema_decay\n        self.student_view_noise_std = student_view_noise_std\n        self.student_view_dropout_prob = student_view_dropout_prob\n\n    @staticmethod\n    def _clone_teacher(student: nn.Module) -> nn.Module:\n        import copy\n\n        teacher = copy.deepcopy(student)\n        teacher.eval()\n        for parameter in teacher.parameters():\n            parameter.requires_grad_(False)\n        return teacher\n\n    def _make_student_view(self, images: torch.Tensor) -> torch.Tensor:\n        augmented = images.clone()\n        if self.student_view_noise_std > 0:\n            augmented = augmented + torch.randn_like(augmented) * self.student_view_noise_std\n        if self.student_view_dropout_prob > 0:\n            dropout_mask = torch.rand_like(augmented[:, :1]) > self.student_view_dropout_prob\n            augmented = augmented * dropout_mask\n        augmented = augmented.clamp(0.0, 1.0)\n        return augmented\n\n    def training_step(\n        self,\n        labeled_images: torch.Tensor,\n        labeled_masks: torch.Tensor,\n        labeled_text_tokens: Optional[torch.Tensor] = None,\n        labeled_structured_attributes: Optional[torch.Tensor] = None,\n        unlabeled_images: Optional[torch.Tensor] = None,\n        unlabeled_teacher_images: Optional[torch.Tensor] = None,\n        unlabeled_student_images: Optional[torch.Tensor] = None,\n        unlabeled_text_tokens: Optional[torch.Tensor] = None,\n        unlabeled_structured_attributes: Optional[torch.Tensor] = None,\n        unlabeled_force_text_dropout_mask: Optional[torch.Tensor] = None,\n        scaler: Optional[torch.amp.GradScaler] = None,\n        use_amp: bool = False,\n        grad_accum_steps: int = 1,\n        zero_grad: bool = True,\n        step_optimizer: bool = True,\n    ) -> Dict[str, torch.Tensor]:\n        self.student.train()\n        device_type = labeled_images.device.type\n        amp_enabled = use_amp and device_type == "cuda"\n        autocast_context = torch.amp.autocast(device_type="cuda", enabled=True) if amp_enabled else nullcontext()\n\n        with autocast_context:\n            labeled_logits, labeled_aux = self.student(\n                labeled_images,\n                text_tokens=labeled_text_tokens,\n                structured_attributes=labeled_structured_attributes,\n                return_aux=True,\n                update_prototypes=True,\n            )\n\n            unlabeled_logits = None\n            teacher_prediction = None\n            teacher_text_prior = None\n            unlabeled_aux = None\n            if unlabeled_images is not None:\n                teacher_images = unlabeled_teacher_images if unlabeled_teacher_images is not None else unlabeled_images\n                student_images = (\n                    unlabeled_student_images if unlabeled_student_images is not None else self._make_student_view(unlabeled_images)\n                )\n                unlabeled_logits, unlabeled_aux = self.student(\n                    student_images,\n                    text_tokens=unlabeled_text_tokens,\n                    structured_attributes=unlabeled_structured_attributes,\n                    return_aux=True,\n                    update_prototypes=False,\n                    force_text_dropout_mask=unlabeled_force_text_dropout_mask,\n                )\n                with torch.no_grad():\n                    self.teacher.eval()\n                    teacher_logits, teacher_aux = self.teacher(\n                        teacher_images,\n                        text_tokens=unlabeled_text_tokens,\n                        structured_attributes=unlabeled_structured_attributes,\n                        return_aux=True,\n                        update_prototypes=False,\n                    )\n                    teacher_prediction = torch.sigmoid(teacher_logits)\n                    teacher_text_prior = teacher_aux["prior_prediction"]\n\n            losses = self.loss_fn(\n                labeled_logits=labeled_logits,\n                labeled_target=labeled_masks,\n                unlabeled_logits=unlabeled_logits,\n                visual_teacher_prediction=teacher_prediction,\n                text_prior_prediction=teacher_text_prior,\n                visual_summary=unlabeled_aux["fused_summary"] if unlabeled_aux is not None else labeled_aux["fused_summary"],\n                prototype_summary=(\n                    unlabeled_aux["prototype_summary"] if unlabeled_aux is not None else labeled_aux["prototype_summary"]\n                ),\n            )\n\n        if zero_grad:\n            self.optimizer.zero_grad(set_to_none=True)\n\n        backward_loss = losses["total"] / max(int(grad_accum_steps), 1)\n        if scaler is not None and amp_enabled:\n            scaler.scale(backward_loss).backward()\n        else:\n            backward_loss.backward()\n\n        if step_optimizer:\n            if scaler is not None and amp_enabled:\n                scaler.step(self.optimizer)\n                scaler.update()\n            else:\n                self.optimizer.step()\n            update_ema_model(self.teacher, self.student, decay=self.ema_decay)\n\n        losses["backward_loss"] = backward_loss.detach()\n        return losses\n'
TRAIN_SCRIPT_SOURCE = 'import argparse\nimport csv\nimport json\nimport random\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport torch\nfrom torch.utils.data import DataLoader, Subset\n\nfrom Load_Dataset import ImageToImage2D, RandomGenerator, ValGenerator\nfrom nets.LViT_improved import LViTImproved\nfrom nets.improved_training import ImprovedSSLTrainer\nfrom text_encoder import (\n    CachedDomainTextEncoder,\n    attribute_vector_size,\n    build_cache_metadata,\n    save_report_features,\n)\nfrom utils import dice_on_batch, read_text\n\n\ndef count_trainable_parameters(model: torch.nn.Module) -> int:\n    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)\n\n\ndef save_json(path: Path, payload: dict) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8") as handle:\n        json.dump(payload, handle, indent=2, ensure_ascii=False)\n\n\ndef save_history(history: list[dict], save_dir: Path) -> None:\n    if not history:\n        return\n    csv_path = save_dir / "history.csv"\n    json_path = save_dir / "history.json"\n    fieldnames = list(history[0].keys())\n    with csv_path.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fieldnames)\n        writer.writeheader()\n        writer.writerows(history)\n    save_json(json_path, {"history": history})\n\n\ndef plot_history(history: list[dict], save_dir: Path) -> None:\n    if not history:\n        return\n\n    epochs = [entry["epoch"] for entry in history]\n    train_loss = [entry["train_total"] for entry in history]\n    val_dice = [entry["val_dice"] for entry in history]\n    lr = [entry["lr"] for entry in history]\n    supervised = [entry["train_supervised"] for entry in history]\n    consistency = [entry["train_consistency"] for entry in history]\n\n    figure, axes = plt.subplots(1, 2, figsize=(12, 4))\n\n    axes[0].plot(epochs, train_loss, label="train_total")\n    axes[0].plot(epochs, supervised, label="train_supervised")\n    axes[0].plot(epochs, consistency, label="train_consistency")\n    axes[0].set_title("Training Losses")\n    axes[0].set_xlabel("Epoch")\n    axes[0].set_ylabel("Loss")\n    axes[0].legend()\n\n    axes[1].plot(epochs, val_dice, label="val_dice")\n    axes[1].plot(epochs, lr, label="lr")\n    axes[1].set_title("Validation / LR")\n    axes[1].set_xlabel("Epoch")\n    axes[1].legend()\n\n    figure.tight_layout()\n    figure.savefig(save_dir / "training_curves.png", dpi=180, bbox_inches="tight")\n    plt.close(figure)\n\n\ndef save_validation_examples(\n    model: torch.nn.Module,\n    loader: DataLoader,\n    device: torch.device,\n    save_dir: Path,\n    epoch: int,\n    max_examples: int = 4,\n) -> None:\n    if max_examples <= 0:\n        return\n\n    example_dir = save_dir / "val_examples"\n    example_dir.mkdir(parents=True, exist_ok=True)\n\n    model.eval()\n    saved = 0\n    with torch.no_grad():\n        for batch in loader:\n            sample, names = batch\n            image = sample["image"].to(device)\n            label = sample["label"]\n            text = sample.get("text")\n            attributes = sample.get("attributes")\n            if text is not None:\n                text = text.to(device)\n            if attributes is not None:\n                attributes = attributes.to(device)\n\n            logits = model(image, text_tokens=text, structured_attributes=attributes)\n            prediction = torch.sigmoid(logits).cpu()\n\n            batch_size = image.shape[0]\n            for index in range(batch_size):\n                if saved >= max_examples:\n                    return\n                figure, axes = plt.subplots(1, 3, figsize=(9, 3))\n                image_np = image[index].detach().cpu().permute(1, 2, 0).numpy()\n                if image_np.shape[-1] == 1:\n                    axes[0].imshow(image_np[..., 0], cmap="gray")\n                else:\n                    axes[0].imshow(image_np)\n                axes[0].set_title("Image")\n                axes[0].axis("off")\n\n                axes[1].imshow(label[index].cpu().numpy(), cmap="gray")\n                axes[1].set_title("GT")\n                axes[1].axis("off")\n\n                axes[2].imshow(prediction[index, 0].numpy(), cmap="viridis", vmin=0.0, vmax=1.0)\n                axes[2].set_title("Pred")\n                axes[2].axis("off")\n\n                figure.tight_layout()\n                name = Path(names[index]).stem\n                figure.savefig(\n                    example_dir / f"epoch_{epoch:03d}_{name}.png",\n                    dpi=180,\n                    bbox_inches="tight",\n                )\n                plt.close(figure)\n                saved += 1\n\n\ndef checkpoint_payload(\n    model: LViTImproved,\n    trainer: ImprovedSSLTrainer,\n    optimizer: torch.optim.Optimizer,\n    scheduler,\n    scaler: torch.amp.GradScaler | None,\n    epoch: int,\n    best_val: float,\n    history: list[dict],\n    config_payload: dict,\n) -> dict:\n    return {\n        "epoch": epoch,\n        "best_val_dice": best_val,\n        "model_state": model.state_dict(),\n        "teacher_state": trainer.teacher.state_dict(),\n        "optimizer_state": optimizer.state_dict(),\n        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,\n        "scaler_state": scaler.state_dict() if scaler is not None else None,\n        "history": history,\n        "config": config_payload,\n    }\n\n\ndef load_checkpoint(\n    checkpoint_path: Path,\n    model: LViTImproved,\n    trainer: ImprovedSSLTrainer,\n    optimizer: torch.optim.Optimizer,\n    scheduler,\n    scaler: torch.amp.GradScaler | None,\n):\n    checkpoint = torch.load(checkpoint_path, map_location="cpu")\n    model.load_state_dict(checkpoint["model_state"])\n    trainer.teacher.load_state_dict(checkpoint["teacher_state"])\n    optimizer.load_state_dict(checkpoint["optimizer_state"])\n    if scheduler is not None and checkpoint.get("scheduler_state") is not None:\n        scheduler.load_state_dict(checkpoint["scheduler_state"])\n    if scaler is not None and checkpoint.get("scaler_state") is not None:\n        scaler.load_state_dict(checkpoint["scaler_state"])\n    history = checkpoint.get("history", [])\n    start_epoch = int(checkpoint.get("epoch", 0))\n    best_val = float(checkpoint.get("best_val_dice", 0.0))\n    return start_epoch, best_val, history\n\n\ndef precompute_text_cache(\n    report_map: dict[str, str],\n    cache_dir: str,\n    max_text_units: int,\n    model_name: str,\n    local_files_only: bool = False,\n) -> None:\n    cache_path = Path(cache_dir)\n    metadata = build_cache_metadata(model_name=model_name, max_units=max_text_units)\n    missing_items = []\n    for sample_id in report_map:\n        cache_file = cache_path / f"{Path(sample_id).stem}.pt"\n        if not cache_file.exists():\n            missing_items.append(sample_id)\n            continue\n        payload = torch.load(cache_file, map_location="cpu", weights_only=True)\n        if payload.get("metadata") != metadata:\n            missing_items.append(sample_id)\n\n    if not missing_items:\n        return\n\n    encoder = CachedDomainTextEncoder(model_name=model_name, local_files_only=local_files_only)\n    for sample_id in missing_items:\n        report = report_map[sample_id]\n        text, attributes = encoder.encode_report(report, max_lines=max_text_units)\n        save_report_features(cache_dir, sample_id, text, attributes, metadata=metadata)\n\n\ndef build_datasets(\n    dataset_root: str,\n    task_name: str,\n    image_size: int,\n    text_model_name: str,\n    max_text_units: int,\n    cache_root: str | None = None,\n    local_files_only: bool = False,\n):\n    train_tf = RandomGenerator(output_size=[image_size, image_size])\n    val_tf = ValGenerator(output_size=[image_size, image_size])\n\n    train_text = read_text(f"{dataset_root}/Train_Folder/Train_text.xlsx")\n    val_text = read_text(f"{dataset_root}/Val_Folder/Val_text.xlsx")\n    if cache_root is None:\n        cache_base = Path(dataset_root)\n        train_cache = str(cache_base / "Train_Folder" / "text_cache")\n        val_cache = str(cache_base / "Val_Folder" / "text_cache")\n    else:\n        cache_base = Path(cache_root)\n        train_cache = str(cache_base / "Train_Folder")\n        val_cache = str(cache_base / "Val_Folder")\n    cache_metadata = build_cache_metadata(model_name=text_model_name, max_units=max_text_units)\n    precompute_text_cache(\n        train_text,\n        train_cache,\n        max_text_units=max_text_units,\n        model_name=text_model_name,\n        local_files_only=local_files_only,\n    )\n    precompute_text_cache(\n        val_text,\n        val_cache,\n        max_text_units=max_text_units,\n        model_name=text_model_name,\n        local_files_only=local_files_only,\n    )\n\n    train_dataset = ImageToImage2D(\n        f"{dataset_root}/Train_Folder/",\n        task_name,\n        train_text,\n        train_tf,\n        image_size=image_size,\n        cache_dir=train_cache,\n        text_model_name=text_model_name,\n        local_files_only=local_files_only,\n        cache_metadata=cache_metadata,\n        max_text_units=max_text_units,\n    )\n    val_dataset = ImageToImage2D(\n        f"{dataset_root}/Val_Folder/",\n        task_name,\n        val_text,\n        val_tf,\n        image_size=image_size,\n        cache_dir=val_cache,\n        text_model_name=text_model_name,\n        local_files_only=local_files_only,\n        cache_metadata=cache_metadata,\n        max_text_units=max_text_units,\n    )\n    return train_dataset, val_dataset\n\n\ndef split_labeled_unlabeled(dataset, label_ratio: float, seed: int):\n    indices = list(range(len(dataset)))\n    rng = random.Random(seed)\n    rng.shuffle(indices)\n    labeled_count = max(1, int(len(indices) * label_ratio))\n    labeled_indices = indices[:labeled_count]\n    unlabeled_indices = indices[labeled_count:]\n    labeled_subset = Subset(dataset, labeled_indices)\n    unlabeled_subset = Subset(dataset, unlabeled_indices) if unlabeled_indices else None\n    return labeled_subset, unlabeled_subset\n\n\ndef resolve_subset_names(subset: Subset | None) -> list[str]:\n    if subset is None:\n        return []\n    dataset = subset.dataset\n    if hasattr(dataset, "images_list"):\n        source_names = dataset.images_list\n    elif hasattr(dataset, "mask_list"):\n        source_names = dataset.mask_list\n    else:\n        return [str(index) for index in subset.indices]\n    return [str(source_names[index]) for index in subset.indices]\n\n\ndef move_batch_to_device(batch, device: torch.device):\n    sample, _ = batch\n    moved = {}\n    for key, value in sample.items():\n        moved[key] = value.to(device)\n    return moved\n\n\ndef infinite_loader(loader: Iterable):\n    while True:\n        yield from loader\n\n\ndef validate(model: torch.nn.Module, loader: DataLoader, device: torch.device) -> float:\n    model.eval()\n    dices = []\n    with torch.no_grad():\n        for batch in loader:\n            sample = move_batch_to_device(batch, device)\n            logits = model(\n                sample["image"],\n                text_tokens=sample.get("text"),\n                structured_attributes=sample.get("attributes"),\n            )\n            preds = torch.sigmoid(logits)\n            dices.append(dice_on_batch(sample["label"], preds))\n    return float(sum(dices) / max(len(dices), 1))\n\n\ndef build_scheduler(optimizer: torch.optim.Optimizer, scheduler_name: str, epochs: int, min_lr: float):\n    if scheduler_name == "none":\n        return None\n    if scheduler_name == "cosine":\n        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1), eta_min=min_lr)\n    raise ValueError(f"Unsupported scheduler: {scheduler_name}")\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--dataset-root", type=str, required=True)\n    parser.add_argument("--task-name", type=str, default="MoNuSeg")\n    parser.add_argument("--epochs", type=int, default=50)\n    parser.add_argument("--batch-size", type=int, default=4)\n    parser.add_argument("--grad-accum-steps", type=int, default=4)\n    parser.add_argument("--label-ratio", type=float, default=0.25)\n    parser.add_argument("--image-size", type=int, default=224)\n    parser.add_argument("--lr", type=float, default=1e-4)\n    parser.add_argument("--min-lr", type=float, default=1e-6)\n    parser.add_argument("--weight-decay", type=float, default=1e-4)\n    parser.add_argument("--scheduler", type=str, choices=["none", "cosine"], default="cosine")\n    parser.add_argument("--seed", type=int, default=666)\n    parser.add_argument("--num-workers", type=int, default=2)\n    parser.add_argument("--ema-decay", type=float, default=0.99)\n    parser.add_argument(\n        "--text-model-name",\n        type=str,\n        default="microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",\n    )\n    parser.add_argument("--max-text-units", type=int, default=10)\n    parser.add_argument("--local-files-only", action="store_true")\n    parser.add_argument("--amp", action=argparse.BooleanOptionalAction, default=True)\n    parser.add_argument("--save-every", type=int, default=0)\n    parser.add_argument("--save-val-examples", type=int, default=4)\n    parser.add_argument("--save-dir", type=str, default="./runs/improved_ssl")\n    parser.add_argument("--cache-root", type=str, default=None)\n    parser.add_argument("--resume", type=str, default=None)\n    args = parser.parse_args()\n\n    random.seed(args.seed)\n    torch.manual_seed(args.seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(args.seed)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    pin_memory = torch.cuda.is_available()\n    amp_enabled = bool(args.amp and torch.cuda.is_available())\n    effective_batch_size = int(args.batch_size) * int(args.grad_accum_steps)\n\n    save_dir = Path(args.save_dir)\n    save_dir.mkdir(parents=True, exist_ok=True)\n    checkpoint_dir = save_dir / "checkpoints"\n    checkpoint_dir.mkdir(parents=True, exist_ok=True)\n\n    train_dataset, val_dataset = build_datasets(\n        args.dataset_root,\n        args.task_name,\n        args.image_size,\n        text_model_name=args.text_model_name,\n        max_text_units=args.max_text_units,\n        cache_root=args.cache_root or str(save_dir / "text_cache"),\n        local_files_only=args.local_files_only,\n    )\n    labeled_dataset, unlabeled_dataset = split_labeled_unlabeled(train_dataset, args.label_ratio, args.seed)\n    split_manifest = {\n        "task_name": args.task_name,\n        "label_ratio": args.label_ratio,\n        "seed": args.seed,\n        "train_dataset_size": len(train_dataset),\n        "val_dataset_size": len(val_dataset),\n        "labeled_count": len(labeled_dataset),\n        "unlabeled_count": 0 if unlabeled_dataset is None else len(unlabeled_dataset),\n        "labeled_samples": resolve_subset_names(labeled_dataset),\n        "unlabeled_samples": resolve_subset_names(unlabeled_dataset),\n    }\n    save_json(save_dir / "split_manifest.json", split_manifest)\n\n    labeled_loader = DataLoader(\n        labeled_dataset,\n        batch_size=args.batch_size,\n        shuffle=True,\n        num_workers=args.num_workers,\n        pin_memory=pin_memory,\n    )\n    unlabeled_loader = None\n    if unlabeled_dataset is not None and len(unlabeled_dataset) > 0:\n        unlabeled_loader = DataLoader(\n            unlabeled_dataset,\n            batch_size=args.batch_size,\n            shuffle=True,\n            num_workers=args.num_workers,\n            pin_memory=pin_memory,\n        )\n    val_loader = DataLoader(\n        val_dataset,\n        batch_size=args.batch_size,\n        shuffle=False,\n        num_workers=args.num_workers,\n        pin_memory=pin_memory,\n    )\n\n    sample_batch = next(iter(labeled_loader))[0]\n    text_dim = sample_batch["text"].shape[-1]\n    model = LViTImproved(\n        n_channels=sample_batch["image"].shape[1],\n        n_classes=1,\n        img_size=args.image_size,\n        text_dim=text_dim,\n        attribute_dim=attribute_vector_size(),\n    ).to(device)\n\n    optimizer = torch.optim.AdamW(\n        model.parameters(),\n        lr=args.lr,\n        weight_decay=args.weight_decay,\n    )\n    scheduler = build_scheduler(optimizer, args.scheduler, args.epochs, args.min_lr)\n    trainer = ImprovedSSLTrainer(student=model, optimizer=optimizer, ema_decay=args.ema_decay)\n    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)\n    unlabeled_iterator = infinite_loader(unlabeled_loader) if unlabeled_loader is not None else None\n\n    config_payload = {\n        **vars(args),\n        "device": str(device),\n        "amp_enabled": amp_enabled,\n        "effective_batch_size": effective_batch_size,\n        "train_dataset_size": len(train_dataset),\n        "val_dataset_size": len(val_dataset),\n        "labeled_dataset_size": len(labeled_dataset),\n        "unlabeled_dataset_size": 0 if unlabeled_dataset is None else len(unlabeled_dataset),\n        "text_dim": int(text_dim),\n        "trainable_parameters": count_trainable_parameters(model),\n    }\n    save_json(save_dir / "run_config.json", config_payload)\n\n    history: list[dict] = []\n    best_val = 0.0\n    start_epoch = 0\n    if args.resume:\n        start_epoch, best_val, history = load_checkpoint(\n            Path(args.resume),\n            model=model,\n            trainer=trainer,\n            optimizer=optimizer,\n            scheduler=scheduler,\n            scaler=scaler,\n        )\n\n    for epoch in range(start_epoch, args.epochs):\n        epoch_metric_lists: dict[str, list[float]] = {\n            "total": [],\n            "supervised": [],\n            "consistency": [],\n            "text_guidance": [],\n            "boundary": [],\n            "prototype": [],\n            "fusion_weight": [],\n        }\n\n        num_labeled_batches = len(labeled_loader)\n        for batch_index, labeled_batch in enumerate(labeled_loader):\n            labeled = move_batch_to_device(labeled_batch, device)\n            unlabeled = None\n            if unlabeled_iterator is not None:\n                unlabeled_batch = next(unlabeled_iterator)\n                unlabeled = move_batch_to_device(unlabeled_batch, device)\n\n            zero_grad = (batch_index % args.grad_accum_steps) == 0\n            step_optimizer = ((batch_index + 1) % args.grad_accum_steps == 0) or ((batch_index + 1) == num_labeled_batches)\n\n            losses = trainer.training_step(\n                labeled_images=labeled["image"],\n                labeled_masks=labeled["label"].float(),\n                labeled_text_tokens=labeled.get("text"),\n                labeled_structured_attributes=labeled.get("attributes"),\n                unlabeled_images=None if unlabeled is None else unlabeled["image"],\n                unlabeled_text_tokens=None if unlabeled is None else unlabeled.get("text"),\n                unlabeled_structured_attributes=None if unlabeled is None else unlabeled.get("attributes"),\n                scaler=scaler,\n                use_amp=amp_enabled,\n                grad_accum_steps=args.grad_accum_steps,\n                zero_grad=zero_grad,\n                step_optimizer=step_optimizer,\n            )\n\n            for key in epoch_metric_lists:\n                epoch_metric_lists[key].append(float(losses[key].item()))\n\n        if scheduler is not None:\n            scheduler.step()\n\n        val_dice = validate(model, val_loader, device)\n        current_lr = float(optimizer.param_groups[0]["lr"])\n        epoch_record = {\n            "epoch": epoch + 1,\n            "train_total": sum(epoch_metric_lists["total"]) / max(len(epoch_metric_lists["total"]), 1),\n            "train_supervised": sum(epoch_metric_lists["supervised"]) / max(len(epoch_metric_lists["supervised"]), 1),\n            "train_consistency": sum(epoch_metric_lists["consistency"]) / max(len(epoch_metric_lists["consistency"]), 1),\n            "train_text_guidance": sum(epoch_metric_lists["text_guidance"]) / max(len(epoch_metric_lists["text_guidance"]), 1),\n            "train_boundary": sum(epoch_metric_lists["boundary"]) / max(len(epoch_metric_lists["boundary"]), 1),\n            "train_prototype": sum(epoch_metric_lists["prototype"]) / max(len(epoch_metric_lists["prototype"]), 1),\n            "train_fusion_weight": sum(epoch_metric_lists["fusion_weight"]) / max(len(epoch_metric_lists["fusion_weight"]), 1),\n            "val_dice": float(val_dice),\n            "lr": current_lr,\n        }\n        history.append(epoch_record)\n        save_history(history, save_dir)\n        plot_history(history, save_dir)\n\n        torch.save(\n            checkpoint_payload(\n                model=model,\n                trainer=trainer,\n                optimizer=optimizer,\n                scheduler=scheduler,\n                scaler=scaler,\n                epoch=epoch + 1,\n                best_val=best_val,\n                history=history,\n                config_payload=config_payload,\n            ),\n            save_dir / "last_model.pt",\n        )\n\n        is_best = val_dice >= best_val\n        if is_best:\n            best_val = float(val_dice)\n            torch.save(\n                checkpoint_payload(\n                    model=model,\n                    trainer=trainer,\n                    optimizer=optimizer,\n                    scheduler=scheduler,\n                    scaler=scaler,\n                    epoch=epoch + 1,\n                    best_val=best_val,\n                    history=history,\n                    config_payload=config_payload,\n                ),\n                save_dir / "best_model.pt",\n            )\n            save_validation_examples(\n                model=model,\n                loader=val_loader,\n                device=device,\n                save_dir=save_dir,\n                epoch=epoch + 1,\n                max_examples=args.save_val_examples,\n            )\n\n        if args.save_every > 0 and (epoch + 1) % args.save_every == 0:\n            torch.save(\n                checkpoint_payload(\n                    model=model,\n                    trainer=trainer,\n                    optimizer=optimizer,\n                    scheduler=scheduler,\n                    scaler=scaler,\n                    epoch=epoch + 1,\n                    best_val=best_val,\n                    history=history,\n                    config_payload=config_payload,\n                ),\n                checkpoint_dir / f"epoch_{epoch + 1:03d}.pt",\n            )\n\n        run_summary = {\n            "best_val_dice": best_val,\n            "last_epoch": epoch + 1,\n            "effective_batch_size": effective_batch_size,\n            "amp_enabled": amp_enabled,\n            "device": str(device),\n        }\n        save_json(save_dir / "run_summary.json", run_summary)\n\n        print(\n            f"epoch={epoch + 1} "\n            f"train_total={epoch_record[\'train_total\']:.4f} "\n            f"train_supervised={epoch_record[\'train_supervised\']:.4f} "\n            f"val_dice={val_dice:.4f} "\n            f"best_val_dice={best_val:.4f} "\n            f"lr={current_lr:.6f}"\n        )\n\n\nif __name__ == "__main__":\n    main()\n'

register_module("utils", UTILS_SOURCE)
register_module("text_encoder", TEXT_ENCODER_SOURCE)
register_module("Load_Dataset", LOAD_DATASET_SOURCE)
register_module("nets.improved_ssl", IMPROVED_SSL_SOURCE, package="nets")
register_module("nets.LViT_improved", IMPROVED_MODEL_SOURCE, package="nets")
register_module("nets.improved_training", IMPROVED_TRAINING_SOURCE, package="nets")
register_module("train_improved_ssl", TRAIN_SCRIPT_SOURCE)

print("Inline modules registered successfully.")


In [ ]:
import os
import sys

os.makedirs(SAVE_DIR, exist_ok=True)

argv = [
    "train_improved_ssl.py",
    "--dataset-root", DATASET_ROOT,
    "--task-name", TASK_NAME,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--grad-accum-steps", str(GRAD_ACCUM_STEPS),
    "--label-ratio", str(LABEL_RATIO),
    "--image-size", str(IMAGE_SIZE),
    "--lr", str(LR),
    "--min-lr", str(MIN_LR),
    "--weight-decay", str(WEIGHT_DECAY),
    "--scheduler", SCHEDULER,
    "--seed", str(SEED),
    "--num-workers", str(NUM_WORKERS),
    "--ema-decay", str(EMA_DECAY),
    "--text-model-name", TEXT_MODEL_NAME,
    "--max-text-units", str(MAX_TEXT_UNITS),
    "--save-every", str(SAVE_EVERY),
    "--save-val-examples", str(SAVE_VAL_EXAMPLES),
    "--save-dir", SAVE_DIR,
]

if AMP:
    argv.append("--amp")
else:
    argv.append("--no-amp")

if LOCAL_FILES_ONLY:
    argv.append("--local-files-only")

print("Running command:")
print(" ".join(argv))

sys.argv = argv
import train_improved_ssl
train_improved_ssl.main()


## Expected outputs

Sau khi train xong, thư mục `SAVE_DIR` sẽ có:

- `best_model.pt`
- `last_model.pt`
- `run_config.json`
- `run_summary.json`
- `split_manifest.json`
- `history.csv`
- `history.json`
- `training_curves.png`
- `val_examples/`